# Vision-Default, Prior-Override: a 5-minute walkthrough

Shown a counterfactual image (a blue banana), a vision-language model must choose
between what it sees (blue) and what it knows (bananas are usually yellow). This
notebook reproduces the paper's core finding on one example with Qwen2.5-VL 3B:
under a prior-knowledge prompt the model reports the memorized color, and ablating
a small set of "promoting" attention heads flips the answer back to the color in
the image. Vision is the default pathway; the prior has to be actively injected.

**Requirements:** a GPU (CUDA, or Apple Silicon via MPS, see the next cell) and a
HuggingFace token. Copy `.env.example` to `.env` and set `HF_TOKEN` (needed for the
gated Qwen weights and the dataset). Models load in 4-bit by default. Run `uv sync` first.

Paper: link added on publication.

### Running on Apple Silicon (MPS)

The model-load cell below targets a CUDA GPU with 4-bit quantization (bitsandbytes is
CUDA-only). On an Apple Silicon Mac, run the 3B model full precision on the MPS backend
instead. Set `PYTORCH_ENABLE_MPS_FALLBACK=1` before launching Jupyter (it lets the few ops
without an MPS kernel fall back to CPU), then replace the next cell with:

```python
mwp = load_vlm(
    ModelSize.SMALL_3B,
    model_family=ModelFamily.QWEN,
    load_in_4bit=False,            # bitsandbytes 4-bit is CUDA-only
    device_map=None,
    attn_implementation="eager",   # most MPS-compatible
)
mwp.model.to("mps")
model, processor, adapter = mwp.model, mwp.processor, mwp.adapter
```

Validated on an M4 (24 GB): the whole walkthrough runs in a couple of minutes and produces
the same Yellow to Blue flip.

In [ ]:
import sys
from pathlib import Path

# Make the vdpo package importable when running from the repo root.
sys.path.insert(0, str(Path.cwd() / "src"))

import torch

from vdpo.types.enums import GroundingMode, ModelFamily, ModelSize
from vdpo.utils.load_model import load_vlm

mwp = load_vlm(ModelSize.SMALL_3B, model_family=ModelFamily.QWEN)
model, processor, adapter = mwp.model, mwp.processor, mwp.adapter

## The conflict

We use a counterfactual image and two prompts:

- **Visual:** "What color is this {object} here?" asks for what the model sees.
- **Prior:** "What color is {a object} usually?" asks for memorized knowledge.

This walkthrough uses the **prior** prompt, where the memorized color wins by default.

In [ ]:
from vdpo.utils.load_examples import load_color_examples

examples = load_color_examples()

# Pick a vivid object; falls back to the first example if it is not present.
# Object names are stored capitalized, so SHOWCASE_OBJECT must be too (e.g. "Banana").
# Print [e.object for e in examples] to see the full list and swap this if the
# baseline below does not cleanly report the original color.
SHOWCASE_OBJECT = "Banana"
example = next((e for e in examples if e.object == SHOWCASE_OBJECT), examples[0])

original_color = example.correct_answer[0]   # the usual / real-world color
counterfact_color = example.incorrect_answer  # the manipulated image color
print("object:              ", example.object)
print("original color:      ", original_color)
print("counterfactual color:", counterfact_color)

example.counterfact_image.pil_image  # displays the counterfactual image

In [ ]:
from vdpo.runner.core import VMIRunnerBase

question_prior = VMIRunnerBase._get_question(example, GroundingMode.PRIOR)
print("prior prompt:", question_prior)

image = example.counterfact_image.pil_image
inputs_prior = adapter.build_inputs(
    processor=processor,
    image=image,
    question=question_prior,
    device=model.device,
)

## Baseline: what does it say under the prior prompt?

We read the next-token logits and compare the two candidate color tokens. This is
the exact quantity the paper measures.

In [ ]:
def color_token_id(color):
    # First sub-token id of the capitalized color, matching the runner's mapping.
    ids = processor.tokenizer(
        text=color.capitalize(), add_special_tokens=False, return_tensors="pt"
    ).input_ids
    return ids[0, 0].item()


id_original = color_token_id(original_color)
id_counterfact = color_token_id(counterfact_color)


@torch.no_grad()
def last_token_logits(inputs):
    return model(**inputs).logits[0, -1]


def report(logits, label):
    top_id = logits.argmax().item()
    top_token = processor.tokenizer.decode([top_id]).strip()
    print(
        f"{label}: top -> {top_token!r}  "
        f"(logit {original_color}={logits[id_original]:.2f} | "
        f"{counterfact_color}={logits[id_counterfact]:.2f})"
    )


base_logits = last_token_logits(inputs_prior)
report(base_logits, "baseline ")

## The intervention

The paper classifies a small set of "promoting" heads that inject the prior. For
Qwen2.5-VL 3B these are the 9 heads below, derived with
`scripts/analysis/classify_attn_heads.py` (the full classification is written to
`data/classify_attn_heads.json`). We zero their contribution at the last token,
exactly as the `KnockoutAttnHeads` runner does.

In [ ]:
from vdpo.interventions import head_knockout

# Promoting heads for qwen/3B (prior circuit).
PROMOTING_HEADS = [
    (26, 0), (26, 5), (26, 6),
    (27, 1), (27, 4),
    (28, 3),
    (31, 3), (31, 7),
    (34, 14),
]

with head_knockout(model, adapter, PROMOTING_HEADS):
    knockout_logits = last_token_logits(inputs_prior)

report(base_logits, "baseline ")
report(knockout_logits, "knockout ")

## The same flip, in words

The logit flip is the measurement; here is the model actually answering. We drop
the "color name only" instruction so it replies in a short sentence.

In [ ]:
question_generate = question_prior.replace(" Respond with the color name only.", "")
inputs_generate = adapter.build_inputs(
    processor=processor,
    image=image,
    question=question_generate,
    device=model.device,
)


@torch.no_grad()
def generate_text(inputs, max_new_tokens=40):
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    return processor.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


print("baseline:", generate_text(inputs_generate))
with head_knockout(model, adapter, PROMOTING_HEADS):
    print("knockout:", generate_text(inputs_generate))

## Recap and next steps

Under the prior prompt the model reported the memorized color; ablating the
promoting heads flipped both the logits and the generated answer to the color in
the image. Vision is the default; the prior is an active override.

To go further:

- Repeat with the **visual** prompt (`GroundingMode.VISUAL`) and confirm ablation
  barely moves it, the asymmetry at the heart of the paper.
- Try other families and sizes (`ModelFamily`, `ModelSize`).
- Reproduce the full pipeline (patching, classification, knockout ablation,
  figures) with the commands in the README.